# 02. Ingest Unstructured Texts (PDFs)
Uses PyPDF and a PySpark UDF to read Ayurvedic theory PDFs and chunks the text. (Fixed `ai_parse_document` error)

In [0]:
# MAGIC %pip install pypdf

In [0]:
from pyspark.sql.functions import col, udf, explode, lit, length
from pyspark.sql.types import ArrayType, StringType, StructType, StructField
import io

# --- Configuration ---
CATALOG = "main"
SCHEMA = "ayurgenix"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/files"
PDF_PATH = f"{VOLUME_PATH}/*.pdf" # Reads all PDFs in the volume
DELTA_TABLE = f"{CATALOG}.{SCHEMA}.ayurgenix_unstructured"

In [0]:
def extract_text_from_pdf(content):
    try:
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(content))
        text_chunks = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                # Basic chunking by paragraph (splitting by double newline)
                chunks = [t.strip() for t in text.split('\n\n') if len(t.strip()) > 50]
                text_chunks.extend(chunks)
        return text_chunks
    except Exception as e:
        return []

# Register the Python function as a Spark UDF
extract_text_udf = udf(extract_text_from_pdf, ArrayType(StringType()))

In [0]:
print("📚 Processing Ayurvedic PDFs using PyPDF...")

try:
    # Read raw PDFs in binary format
    raw_pdf_df = spark.read.format("binaryFile").load(PDF_PATH)
    
    # If using DB14.3+, checking count may fail if path doesn't exist, wrapped in try-except
    if raw_pdf_df.count() == 0:
        raise ValueError("No PDFs found")
        
    # Extract text chunks using the UDF
    pdf_extracted_df = raw_pdf_df.withColumn("chunks", extract_text_udf("content"))
    
    # Explode array of chunks into individual rows
    pdf_chunks_df = (
        pdf_extracted_df
        .select(
            lit("PDF").alias("source"),
            lit("Ayurvedic_Theory").alias("topic"),
            explode(col("chunks")).alias("text_chunk")
        )
        .filter(length("text_chunk") > 50) # Remove noise and very short bursts
    )
    print(f"✅ Extracted {pdf_chunks_df.count()} text chunks from PDFs.")

except Exception as e:
    print(f"⚠️ PDF Processing failed (possibly no files found?): {e}")
    # Create empty dataframe with correct schema 
    schema = StructType([
        StructField("source", StringType(), True),
        StructField("topic", StringType(), True),
        StructField("text_chunk", StringType(), True)
    ])
    pdf_chunks_df = spark.createDataFrame([], schema)

In [0]:
print(f"💾 Saving chunks to {DELTA_TABLE}...")

pdf_chunks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(DELTA_TABLE)

display(spark.table(DELTA_TABLE).limit(5))

**Next Step:** Run `03_create_master_table` to combine data and log to MLflow.

In [0]:
# COMMAND ----------
def extract_text_from_pdf(content):
    try:
        from pypdf import PdfReader
        import io
        
        reader = PdfReader(io.BytesIO(content))
        text_chunks = []
        
        for page in reader.pages:
            text = page.extract_text()
            if text:
                # Clean up excess whitespace
                clean_text = " ".join(text.split())
                
                # If text is found, chunk roughly every 500 characters
                # instead of relying on \n\n (which often breaks in PDFs)
                chunk_size = 500
                for i in range(0, len(clean_text), chunk_size):
                    chunk = clean_text[i:i+chunk_size].strip()
                    if len(chunk) > 30: # more lenient length filter
                        text_chunks.append(chunk)

        if not text_chunks:
            return ["WARNING: pypdf ran but found no text (Is this a scanned image PDF?)"]
            
        return text_chunks
        
    except Exception as e:
        return [f"ERROR: {str(e)}"]

# Register the Python function as a Spark UDF
extract_text_udf = udf(extract_text_from_pdf, ArrayType(StringType()))
